# EVM CUDA Benchmark on Google ColabThis notebook builds the CUDA-accelerated Eulerian Video Magnification pipeline from scratch, profiles both the **color** (pulse) and **motion** (breathing) pipelines in **FP32 and FP16**, and renders all 4 output videos.**Requirements:** Colab with a GPU runtime (T4 or P100, free tier works).The full technical writeup is in the [repository blog post](https://github.com/iamkucuk/eulerian-video-magnification-cuda/blob/main/docs/blog_speedup.md).

## 1. Environment setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

In [ ]:
# Install build dependencies
!pip install -q cmake ninja pybind11 numpy scipy opencv-python requests

In [ ]:
import os, subprocess

# Clone the repository
if not os.path.exists('evm_cuda'):
    subprocess.run(['git', 'clone',
        'https://github.com/iamkucuk/eulerian-video-magnification-cuda.git',
        'evm_cuda'], check=True)
os.chdir('evm_cuda')
print(f'Working dir: {os.getcwd()}')

## 2. Build the CUDA extensionAuto-detects the GPU's compute capability for a fast single-arch build.

In [ ]:
import subprocess, os

# Detect GPU compute capability
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True, check=True)
cuda_arch = result.stdout.strip().replace('.', '')
print(f'Detected compute capability: sm_{cuda_arch}')

# Configure pybind11
pybind_dir = subprocess.run(
    ['python3', '-c', 'import pybind11; print(pybind11.get_cmake_dir())'],
    capture_output=True, text=True, check=True).stdout.strip()
os.environ['pybind11_DIR'] = pybind_dir

# Build
build_dir = 'cuda/build'
if os.path.exists(build_dir):
    import shutil; shutil.rmtree(build_dir)
os.makedirs(build_dir)

subprocess.run([
    'cmake', '-S', 'cuda', '-B', build_dir,
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}',
    '-G', 'Ninja'], check=True)
subprocess.run(['cmake', '--build', build_dir, '--config', 'Release', '-j'], check=True)
print('Build complete.')

In [ ]:
# Smoke test: verify the extension loads
import sys
sys.path.insert(0, "cuda")
import evm_cuda
from evm_cuda import _evm_cuda
print(f"CUDA extension loaded. have_cuda={evm_cuda.have_cuda}")
print(f"binom5 filter: {_evm_cuda.binom5()}")
print(f"drop_last: {_evm_cuda.drop_last}")

## 3. Download sample videosFetches `face.mp4` (pulse/color) and `baby.mp4` (breathing/motion) from MIT CSAIL.

In [ ]:
subprocess.run(['python', 'scripts/download_samples.py', 'face', 'baby'], check=True)

import os
for f in ['data/face.mp4', 'data/baby.mp4']:
    size = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

## 4. GPU pipeline profiling (FP32 + FP16)Profiles all 4 configurations (color FP32, color FP16, motion FP32, motion FP16) with per-stage breakdown.Each stage is timed with `cudaDeviceSynchronize` between stages. 5 iterations with warmup, median reported.> **Note:** FP32 motion requires ~23 GB VRAM. On a 16 GB GPU (T4/P100), it may OOM — the profiler catches this and continues with FP16 only.

In [ ]:
import gc, json, time
import numpy as np
from evm_cuda import _evm_cuda
from evm_cuda.batched import (DeviceBuffer, _read_frames, _d_binom5,
                               _d_binom5_sum1, _warmup_gpu_pool,
                               _warmup_gpu_pool_motion, figure6_alpha_schedule)

def sync():
    _evm_cuda.device_synchronize()

def gpu_mem_gb():
    """Free / total GPU memory in GB."""
    free_b, total_b = _evm_cuda.gpu_mem_info()
    return free_b / 1e9, total_b / 1e9

def median(xs):
    s = sorted(xs); n = len(s)
    return s[n//2] if n % 2 else (s[n//2-1] + s[n//2]) / 2

def time_stages(run_once, n_iter=5):
    run_once()  # warmup
    runs = [run_once() for _ in range(n_iter)]
    gc.collect()
    return {k: median([r[k] for r in runs]) for k in runs[0]}

results = {}

# --- Setup (shared by FP32 and FP16) ---
frames, fps = _read_frames('data/face.mp4')
n = len(frames); h, w = frames[0].shape[:2]
clip = np.stack(frames, axis=0)
hl, wl = h, w
for _ in range(4): hl = (hl+1)//2; wl = (wl+1)//2
ntsc_floats = n * h * w * 3
print(f'face.mp4: {n} frames, {w}x{h}, filt level: {wl}x{hl}')
free_gb, total_gb = gpu_mem_gb()
print(f'GPU memory before warmup: {free_gb:.1f} GB free / {total_gb:.1f} GB total')
_warmup_gpu_pool(); sync()
free_gb, _ = gpu_mem_gb()
print(f'GPU memory after warmup:  {free_gb:.1f} GB free')
d_clip = DeviceBuffer.from_array(clip)
d_out = DeviceBuffer(n * h * w * 3)
d_filt = DeviceBuffer.from_array(np.zeros((n, hl, wl, 3), dtype=np.float32))
del frames, clip
free_gb, _ = gpu_mem_gb()
print(f'GPU memory after alloc:   {free_gb:.1f} GB free')

# --- Color FP32 ---
print('\n' + '=' * 50)
print('COLOR FP32')
print('=' * 50)
try:
    def run_once():
        st = {}
        t0 = time.perf_counter()
        ntsc = DeviceBuffer(n * h * w * 3 * 4)
        _evm_cuda.batched_bgr_u8_to_ntsc_f32(d_clip.ptr, ntsc.ptr, n, h, w); sync()
        st['1) color_cvt'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        planar = DeviceBuffer(n * 3 * h * w * 4)
        _evm_cuda.batched_to_planar_3ch(ntsc.ptr, planar.ptr, n, h, w)
        gdown = DeviceBuffer(n * 3 * hl * wl * 4)
        _evm_cuda.batched_blur_dn_color(planar.ptr, gdown.ptr, n * 3, h, w, 4, _d_binom5_sum1(), 5); sync()
        st['2) blur_dn'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        _evm_cuda.batched_upsample_add_quantize(
            ntsc.ptr, d_filt.ptr, d_out.ptr, n, hl, wl, h, w, 1.0); sync()
        st['4) render'] = time.perf_counter() - t0
        del ntsc, planar, gdown
        st['_total'] = sum(v for k, v in st.items() if not k.startswith('_'))
        return st
    s = time_stages(run_once)
    results['color_fp32'] = s
    for k, v in s.items():
        if not k.startswith('_'): print(f'  {k:<24s} {v*1000:.1f}ms')
        if k == '_total': print(f'  {"TOTAL":<24s} {v*1000:.1f}ms')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()
gc.collect(); sync()

# --- Color FP16 ---
print('\n' + '=' * 50)
print('COLOR FP16')
print('=' * 50)
free_gb, _ = gpu_mem_gb()
print(f'GPU memory: {free_gb:.1f} GB free')
try:
    def run_once():
        st = {}
        t0 = time.perf_counter()
        ntsc_f32 = DeviceBuffer(ntsc_floats * 4)
        ntsc_f16 = DeviceBuffer(ntsc_floats * 2)
        _evm_cuda.batched_bgr_u8_to_ntsc_f32(d_clip.ptr, ntsc_f32.ptr, n, h, w)
        _evm_cuda.f32_to_f16(ntsc_f32.ptr, ntsc_f16.ptr, ntsc_floats); sync()
        st['1) color_cvt'] = time.perf_counter() - t0
        del ntsc_f32
        t0 = time.perf_counter()
        _evm_cuda.batched_upsample_add_quantize_f16(
            ntsc_f16.ptr, d_filt.ptr, d_out.ptr, n, hl, wl, h, w, 1.0); sync()
        st['4) render'] = time.perf_counter() - t0
        del ntsc_f16
        st['_total'] = sum(v for k, v in st.items() if not k.startswith('_'))
        return st
    s = time_stages(run_once)
    results['color_fp16'] = s
    for k, v in s.items():
        if not k.startswith('_'): print(f'  {k:<24s} {v*1000:.1f}ms')
        if k == '_total': print(f'  {"TOTAL":<24s} {v*1000:.1f}ms')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()
gc.collect(); sync()

## 5. Motion pipeline profiling (FP32 + FP16)Motion magnification: 9-level Laplacian pyramid + IIR temporal filter.FP16 motion fits in ~12 GB VRAM (down from ~23 GB for FP32). On a 16 GB T4/P100, FP32 may OOM.

In [ ]:
# --- Setup (shared by FP32 and FP16) ---
mframes, mfps = _read_frames('data/baby.mp4')
mn = len(mframes); mh, mw = mframes[0].shape[:2]
mclip = np.stack(mframes, axis=0)
levels = 1; hh, ww = mh, mw
while hh >= 5 and ww >= 5: levels += 1; hh = (hh+1)//2; ww = (ww+1)//2
alpha_sched = figure6_alpha_schedule(levels, 10, 16, mh, mw)
lvl_sizes = []; ch, cw = mh, mw
for _ in range(levels): lvl_sizes.append(ch*cw); ch=(ch+1)//2; cw=(cw+1)//2
total_band = sum(s*(mn*3) for s in lvl_sizes)
max_sz = max(lvl_sizes)
level_offsets = []; off = 0
for sz in lvl_sizes: level_offsets.append(off); off += sz*mn*3
ntsc_floats_m = mn*mh*mw*3; planar_floats_m = mn*3*mh*mw
print(f'baby.mp4: {mn} frames, {mw}x{mh}, {levels} pyramid levels')
free_gb, total_gb = gpu_mem_gb()
print(f'GPU memory before warmup: {free_gb:.1f} GB free / {total_gb:.1f} GB total')
_warmup_gpu_pool_motion(mn, mh, mw, levels); sync()
free_gb, _ = gpu_mem_gb()
print(f'GPU memory after warmup:  {free_gb:.1f} GB free')
d_mclip = DeviceBuffer.from_array(mclip)
d_mout = DeviceBuffer(mn*mh*mw*3)
del mframes, mclip
free_gb, _ = gpu_mem_gb()
print(f'GPU memory after alloc:   {free_gb:.1f} GB free')

# --- Motion FP32 ---
print('\n' + '=' * 50)
print('MOTION FP32')
print('=' * 50)
try:
    def run_once():
        st = {}
        t0 = time.perf_counter()
        ntsc = DeviceBuffer(mn*mh*mw*3*4)
        _evm_cuda.batched_bgr_u8_to_ntsc_f32(d_mclip.ptr, ntsc.ptr, mn, mh, mw); sync()
        st['A) NTSC'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        planar = DeviceBuffer(mn*3*mh*mw*4)
        _evm_cuda.batched_to_planar_3ch(ntsc.ptr, planar.ptr, mn, mh, mw)
        bands = DeviceBuffer(total_band*4)
        _evm_cuda.batched_lpyr_build(planar.ptr, bands.ptr, mn, mh, mw, levels, _d_binom5(), 5)
        del planar; sync()
        st['B) lpyr_build'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        filtered = DeviceBuffer(total_band*4)
        nt_buf = DeviceBuffer(mn*max_sz*4); filt_buf = DeviceBuffer(mn*max_sz*4)
        for l in range(levels):
            sz = lvl_sizes[l]
            for c in range(3):
                so = level_offsets[l] + c*mn*sz
                _evm_cuda.batched_thwc_to_nt(bands.ptr_at(so), nt_buf.ptr, mn, sz)
                _evm_cuda.batched_iir_bandpass(nt_buf.ptr, filt_buf.ptr, mn, sz, 0.4, 0.05)
                _evm_cuda.batched_nt_to_thwc_scaled(filt_buf.ptr, filtered.ptr_at(so), mn, sz, alpha_sched[l])
        sync(); st['C) IIR'] = time.perf_counter() - t0
        del bands, nt_buf, filt_buf
        t0 = time.perf_counter()
        delta = DeviceBuffer(mn*3*mh*mw*4)
        _evm_cuda.batched_lpyr_recon(filtered.ptr, delta.ptr, mn, mh, mw, levels, _d_binom5(), 5); sync()
        st['D1) recon'] = time.perf_counter() - t0
        del filtered
        t0 = time.perf_counter()
        _evm_cuda.batched_add_planar_quantize(ntsc.ptr, delta.ptr, d_mout.ptr, mn, mh, mw, 0.1); sync()
        st['D2) render'] = time.perf_counter() - t0
        del ntsc, delta
        st['_total'] = sum(v for k, v in st.items() if not k.startswith('_'))
        return st
    s = time_stages(run_once)
    results['motion_fp32'] = s
    for k, v in s.items():
        if not k.startswith('_'): print(f'  {k:<24s} {v*1000:.1f}ms')
        if k == '_total': print(f'  {"TOTAL":<24s} {v*1000:.1f}ms')
except Exception as e:
    print(f'  FAILED (likely OOM): {e}')
    import traceback; traceback.print_exc()
gc.collect(); sync()

# --- Motion FP16 ---
print('\n' + '=' * 50)
print('MOTION FP16')
print('=' * 50)
free_gb, _ = gpu_mem_gb()
print(f'GPU memory: {free_gb:.1f} GB free')
try:
    def run_once():
        st = {}
        t0 = time.perf_counter()
        ntsc_f32 = DeviceBuffer(ntsc_floats_m*4)
        ntsc_f16 = DeviceBuffer(ntsc_floats_m*2)
        _evm_cuda.batched_bgr_u8_to_ntsc_f32(d_mclip.ptr, ntsc_f32.ptr, mn, mh, mw)
        _evm_cuda.f32_to_f16(ntsc_f32.ptr, ntsc_f16.ptr, ntsc_floats_m); sync()
        st['A) NTSC'] = time.perf_counter() - t0
        del ntsc_f32
        t0 = time.perf_counter()
        planar = DeviceBuffer(planar_floats_m*2)
        _evm_cuda.batched_to_planar_3ch_f16(ntsc_f16.ptr, planar.ptr, mn, mh, mw)
        bands_f32 = DeviceBuffer(total_band*4)
        _evm_cuda.batched_lpyr_build_f16(planar.ptr, bands_f32.ptr, mn, mh, mw, levels, _d_binom5(), 5)
        del planar
        bands = DeviceBuffer(total_band*2)
        _evm_cuda.f32_to_f16(bands_f32.ptr, bands.ptr, total_band)
        del bands_f32; sync()
        st['B) lpyr_build'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        filtered = DeviceBuffer(total_band*2)
        nt_buf = DeviceBuffer(mn*max_sz*2); filt_buf = DeviceBuffer(mn*max_sz*2)
        for l in range(levels):
            sz = lvl_sizes[l]
            for c in range(3):
                so = level_offsets[l] + c*mn*sz
                _evm_cuda.batched_thwc_to_nt_f16(bands.ptr_at_half(so), nt_buf.ptr, mn, sz)
                _evm_cuda.batched_iir_bandpass_f16(nt_buf.ptr, filt_buf.ptr, mn, sz, 0.4, 0.05)
                _evm_cuda.batched_nt_to_thwc_scaled_f16(filt_buf.ptr, filtered.ptr_at_half(so), mn, sz, alpha_sched[l])
        sync(); st['C) IIR'] = time.perf_counter() - t0
        del bands, nt_buf, filt_buf
        t0 = time.perf_counter()
        delta = DeviceBuffer(mn*3*mh*mw*2)
        _evm_cuda.batched_lpyr_recon_f16(filtered.ptr, delta.ptr, mn, mh, mw, levels, _d_binom5(), 5); sync()
        st['D1) recon'] = time.perf_counter() - t0
        del filtered
        t0 = time.perf_counter()
        _evm_cuda.batched_add_planar_quantize_f16(ntsc_f16.ptr, delta.ptr, d_mout.ptr, mn, mh, mw, 0.1); sync()
        st['D2) render'] = time.perf_counter() - t0
        del ntsc_f16, delta
        st['_total'] = sum(v for k, v in st.items() if not k.startswith('_'))
        return st
    s = time_stages(run_once)
    results['motion_fp16'] = s
    for k, v in s.items():
        if not k.startswith('_'): print(f'  {k:<24s} {v*1000:.1f}ms')
        if k == '_total': print(f'  {"TOTAL":<24s} {v*1000:.1f}ms')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()
gc.collect(); sync()

## 6. Render output videosRuns both pipelines (FP32 and FP16) end-to-end and saves results.

In [ ]:
import os, time
os.makedirs('output', exist_ok=True)

from evm_cuda.batched import (magnify_color_gdown_ideal,
                               magnify_color_gdown_ideal_fp16,
                               magnify_motion_lpyr_iir,
                               magnify_motion_lpyr_iir_fp16)

renders = [
    ('face FP32', magnify_color_gdown_ideal,
     dict(vid='data/face.mp4', out='output/face_fp32.mp4',
          alpha=50, level=4, fl=50/60, fh=60/60, chrom_attenuation=1.0, sampling_rate=30.0)),
    ('face FP16', magnify_color_gdown_ideal_fp16,
     dict(vid='data/face.mp4', out='output/face_fp16.mp4',
          alpha=50, level=4, fl=50/60, fh=60/60, chrom_attenuation=1.0, sampling_rate=30.0)),
    ('baby FP32', magnify_motion_lpyr_iir,
     dict(vid='data/baby.mp4', out='output/baby_fp32.mp4',
          alpha=10, lambda_c=16, r1=0.4, r2=0.05, chrom_attenuation=0.1)),
    ('baby FP16', magnify_motion_lpyr_iir_fp16,
     dict(vid='data/baby.mp4', out='output/baby_fp16.mp4',
          alpha=10, lambda_c=16, r1=0.4, r2=0.05, chrom_attenuation=0.1)),
]

for name, fn, params in renders:
    vid = params.pop('vid'); out = params.pop('out')
    try:
        t0 = time.time()
        fn(vid, out, **params)
        size = os.path.getsize(out) / 1024 / 1024
        print(f'  {name}: {size:.1f} MB, {time.time()-t0:.2f}s')
    except Exception as e:
        print(f'  {name}: FAILED ({e})')
    gc.collect()

## 7. Display output videos

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

def show_video(path, label=''):
    if not os.path.exists(path):
        print(f'  {label}: file not found')
        return
    mp4 = open(path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<h4>{label}</h4><video width=480 controls loop><source src="{data_url}" type="video/mp4"></video>'))

for name, path in [('Color FP32 (pulse)', 'output/face_fp32.mp4'),
                   ('Color FP16 (pulse)', 'output/face_fp16.mp4'),
                   ('Motion FP32 (breathing)', 'output/baby_fp32.mp4'),
                   ('Motion FP16 (breathing)', 'output/baby_fp16.mp4')]:
    show_video(path, name)

## 8. Results summary

In [ ]:
# Summary table
print(f'GPU: {subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], capture_output=True, text=True).stdout.strip()}')
print()
print(f'{"Pipeline":<20s} {"FP32":>10s} {"FP16":>10s} {"FP16/FP32":>10s}')
print('-' * 52)
for pipe in ['color', 'motion']:
    fp32 = results.get(f'{pipe}_fp32', {}).get('_total', 0)
    fp16 = results.get(f'{pipe}_fp16', {}).get('_total', 0)
    ratio = f'{fp16/fp32:.2f}x' if fp32 > 0 and fp16 > 0 else '-'
    f32 = f'{fp32*1000:.0f}ms' if fp32 > 0 else 'OOM'
    f16 = f'{fp16*1000:.0f}ms' if fp16 > 0 else 'OOM'
    print(f'{pipe:<20s} {f32:>10s} {f16:>10s} {ratio:>10s}')

print()
print('Reference (A100-80GB):')
print('  Color:  72ms FP32 / 84ms FP16')
print('  Motion: 209ms FP32 / 172ms FP16 (269x vs CPU)')